In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score


In [2]:
df = pd.read_csv('../data/model_data_revised.csv')

In [3]:
df.columns

Index(['tmdb_id', 'imdb_id', 'title_tmdb', 'release_date', 'runtime',
       'release_year', 'n_genres', 'imdb_director_count',
       'dir_prior_success_rate_smoothed', 'dir_prior_films_log1p',
       'dir_years_active_log1p', 'dir_prior_known_films_log1p',
       'lead_actor_films_5yr_backfilled_log1p', 'company_films_5yr_log1p',
       'franchise_prior_films_log1p', 'us_certification', 'release_month',
       'is_sequel', 'is_english', 'dir_first_film', 'dir_no_track_record',
       'backfill_unavailable', 'genre_drama', 'genre_comedy', 'genre_thriller',
       'genre_action', 'genre_adventure', 'genre_crime', 'genre_romance',
       'genre_horror', 'genre_science_fiction', 'genre_family',
       'genre_fantasy', 'genre_mystery', 'genre_animation', 'genre_history',
       'genre_war', 'genre_music', 'genre_western', 'genre_documentary',
       'success'],
      dtype='object')

### Predicting if the movie was a success

In [4]:
#these identifiers need to ne kept out as they cause a data leak / high cardinality
X = df.drop(columns = ['success', 'title_tmdb','tmdb_id', 'imdb_id', 'release_date'])

#needed for the ratings
X = pd.get_dummies(X, columns=['us_certification'], dtype=int)
y = df['success']

#splitting the data for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42,    
    stratify=y          
)

#automatically fitting scaler on training set
print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")


Training features shape: (5535, 41)
Testing features shape: (1384, 41)


In [5]:
print(X.columns.tolist())

['runtime', 'release_year', 'n_genres', 'imdb_director_count', 'dir_prior_success_rate_smoothed', 'dir_prior_films_log1p', 'dir_years_active_log1p', 'dir_prior_known_films_log1p', 'lead_actor_films_5yr_backfilled_log1p', 'company_films_5yr_log1p', 'franchise_prior_films_log1p', 'release_month', 'is_sequel', 'is_english', 'dir_first_film', 'dir_no_track_record', 'backfill_unavailable', 'genre_drama', 'genre_comedy', 'genre_thriller', 'genre_action', 'genre_adventure', 'genre_crime', 'genre_romance', 'genre_horror', 'genre_science_fiction', 'genre_family', 'genre_fantasy', 'genre_mystery', 'genre_animation', 'genre_history', 'genre_war', 'genre_music', 'genre_western', 'genre_documentary', 'us_certification_G', 'us_certification_NR', 'us_certification_PG', 'us_certification_PG-13', 'us_certification_R/NC-17', 'us_certification_Unknown']


In [ ]:
import numpy as np

#finding where there are missing values
X_test.isna().sum()[X_test.isna().sum() > 0]

runtime    3
dtype: int64

In [11]:
df.loc[df['runtime'].isna(), ['title_tmdb', 'runtime']]

,title_tmdb,runtime
2320,Alesha Popovich and Tugarin the Dragon,NaN
2589,Dobrinya and the Dragon,NaN
3298,A.R.O.G,NaN


In [14]:
#filling in the missing values for run time with the median of the training set
runtime_median = X_train['runtime'].median()

X_train['runtime'] = X_train['runtime'].fillna(runtime_median)
X_test['runtime'] = X_test['runtime'].fillna(runtime_median)

In [19]:
X_test['runtime'].isna().sum()

0

In [20]:
#automatically fitting scaler on training set
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter = 1000, random_state = 42,  multi_class='multinomial'))
])
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.6763005780346821
Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.88      0.78       883
           1       0.60      0.32      0.41       501

    accuracy                           0.68      1384
   macro avg       0.65      0.60      0.60      1384
weighted avg       0.66      0.68      0.65      1384



/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1237: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.
  warnings.warn(
